# Question #1: Find πⱼ and Pⱼ

In [1]:
import numpy as np
from collections import Counter
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer

# load training data, only using main content and not headers, footers, etc.
train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))

# calc. πⱼ (priors) for all 20 classes --> prior πⱼ = P(class=j) = fraction of training docs that belong to class j (initial probability before looking at any words)
# Get number of docs in each class, and total amt of training docs
class_counts = Counter(train.target)
total_docs = len(train.target)


# loop through all 20 classes; πⱼ = # docs in class j / total # docs
priors = {}
print("Priors (πⱼ) for each class:\n")

for class_num in range(20):
    count = class_counts[class_num]
    class_name = train.target_names[class_num]
    priors[class_num] = count / total_docs
    print(f"Class {class_num}: P(y={class_name}) = {count}/{total_docs} = {count/total_docs:.4f}")

Priors (πⱼ) for each class:

Class 0: P(y=alt.atheism) = 480/11314 = 0.0424
Class 1: P(y=comp.graphics) = 584/11314 = 0.0516
Class 2: P(y=comp.os.ms-windows.misc) = 591/11314 = 0.0522
Class 3: P(y=comp.sys.ibm.pc.hardware) = 590/11314 = 0.0521
Class 4: P(y=comp.sys.mac.hardware) = 578/11314 = 0.0511
Class 5: P(y=comp.windows.x) = 593/11314 = 0.0524
Class 6: P(y=misc.forsale) = 585/11314 = 0.0517
Class 7: P(y=rec.autos) = 594/11314 = 0.0525
Class 8: P(y=rec.motorcycles) = 598/11314 = 0.0529
Class 9: P(y=rec.sport.baseball) = 597/11314 = 0.0528
Class 10: P(y=rec.sport.hockey) = 600/11314 = 0.0530
Class 11: P(y=sci.crypt) = 595/11314 = 0.0526
Class 12: P(y=sci.electronics) = 591/11314 = 0.0522
Class 13: P(y=sci.med) = 594/11314 = 0.0525
Class 14: P(y=sci.space) = 593/11314 = 0.0524
Class 15: P(y=soc.religion.christian) = 599/11314 = 0.0529
Class 16: P(y=talk.politics.guns) = 546/11314 = 0.0483
Class 17: P(y=talk.politics.mideast) = 564/11314 = 0.0498
Class 18: P(y=talk.politics.misc) = 46

In [2]:
# use CountVectorizer to get all unique words across docs, and get word counts for each doc
vectorizer = CountVectorizer()

# learn from train data, then convert each doc into word count vector
X_train = vectorizer.fit_transform(train.data)

# puts vocab into a word array
vocab = vectorizer.get_feature_names_out()

vocab_size = len(vocab)

print(f"Vocabulary Size/ Unique Words: {vocab_size}")
print(f"X_train Shape (Docs, Words): {X_train.shape}")

Vocabulary Size/ Unique Words: 101631
X_train Shape (Docs, Words): (11314, 101631)


In [3]:
# calc P(w|class=j) for each class j w/ Laplace smoothing. prob. dist. Pⱼ over vocabulary V for each class
word_probs = {}


for class_num in range(20):
    # get all docs in this class
    class_docs = [i for i, label in enumerate(train.target) if label == class_num]

    # concatenate all docs. Sum the word counts across all docs in this class. flatten() converts from a matrix to an array
    class_word_counts = np.array(X_train[class_docs].sum(axis=0)).flatten()

    # use Laplace smoothing (Add 1 to every word count) to avoid zero probabilities
    smoothed_counts = class_word_counts + 1

    # calc total word count w/ smoothing (normalize).
    total_words = smoothed_counts.sum()

    # calc probability for each word --> P(w|class=j) = (count(w in class j) + 1) / (total words in class j + vocab_size)
    word_probs[class_num] = smoothed_counts / total_words


print("Word Probabilities Calculated For All 20 Classes")
print(f"Shape of word_probs[0]: {word_probs[0].shape}")

# example -  look up probability of specific word
if 'computer' in vectorizer.vocabulary_:
    word_idx = vectorizer.vocabulary_['computer']
    print(f"Example: P('computer'|class=0) = {word_probs[0][word_idx]:.6f}")

Word Probabilities Calculated For All 20 Classes
Shape of word_probs[0]: (101631,)
Example: P('computer'|class=0) = 0.000037


# Question #2: Write a routine that uses this naive Bayes model with log probabilities (to avoid underflow) to classify a new document.

In [4]:
# Build classifier function

# Classify a single doc using Multinomial Naive Bayes with log probabilities
def classify_document(doc_text, vectorizer, priors, word_probs):

    # transform doc into word count vector
    doc_counts = vectorizer.transform([doc_text]).toarray().flatten()

    # calc. log prob for each class
    log_probs = np.zeros(20)

    for class_num in range(20):
        # start w/ log prior: log(πⱼ)
        log_probs[class_num] = np.log(priors[class_num])

        # Add log probabilities for each word that appears in doc.  For each word w, add count(w) × log(P(w|class))
        nonzero_idx = doc_counts > 0
        log_probs[class_num] += np.sum(doc_counts[nonzero_idx] * np.log(word_probs[class_num][nonzero_idx]))

    # return class w/ highest log prob
    return np.argmax(log_probs)


# Test on 1st training doc
test_doc = train.data[0]
true_class = train.target[0]

predicted = classify_document(test_doc, vectorizer, priors, word_probs)

print(f"True Class: {true_class} ({train.target_names[true_class]})")
print(f"Predicted Class: {predicted} ({train.target_names[predicted]})")

True Class: 7 (rec.autos)
Predicted Class: 7 (rec.autos)


# Question #3: Evaluate the performance of your model on the test data. What error rate do you achieve?

In [7]:
# Load test data
test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

# Classify all test documents
predictions = []

for doc in test.data:
    pred = classify_document(doc, vectorizer, priors, word_probs)
    predictions.append(pred)

print(f"Classified {len(predictions)} Test Documents")

# Calculate error rate
from sklearn.metrics import accuracy_score, classification_report

# Accuracy
accuracy = accuracy_score(test.target, predictions)
error_rate = 1 - accuracy

print(f"\nAccuracy: {accuracy:.4f}")
print(f"Error Rate: {error_rate:.4f}")

# Detailed classification report
print("\nClassification Report:")
print(classification_report(test.target, predictions, target_names=train.target_names))

Classified 7532 Test Documents

Accuracy: 0.5431
Error Rate: 0.4569

Classification Report:
                          precision    recall  f1-score   support

             alt.atheism       0.65      0.15      0.25       319
           comp.graphics       0.63      0.60      0.62       389
 comp.os.ms-windows.misc       0.33      0.00      0.01       394
comp.sys.ibm.pc.hardware       0.54      0.66      0.60       392
   comp.sys.mac.hardware       0.82      0.42      0.55       385
          comp.windows.x       0.53      0.81      0.64       395
            misc.forsale       0.88      0.55      0.68       390
               rec.autos       0.85      0.54      0.66       396
         rec.motorcycles       0.95      0.40      0.57       398
      rec.sport.baseball       0.97      0.56      0.71       397
        rec.sport.hockey       0.57      0.78      0.66       399
               sci.crypt       0.40      0.79      0.53       396
         sci.electronics       0.70      0.38    